# Gemini Key Verification Notebook

This notebook validates that the configured Gemini API key can initialize the SDK and successfully complete a smoke-test generation request.

## 1. Install and Import Dependencies

In [1]:
import importlib
import platform
import subprocess
import sys

# Install packages if missing (idempotent).
def ensure_package(pkg: str) -> None:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure_package("google-generativeai")
ensure_package("dotenv")

import google.generativeai as genai
from dotenv import load_dotenv
import os
import json
import time
from datetime import datetime, timezone

print("Python:", platform.python_version())
print("google-generativeai:", getattr(genai, "__version__", "unknown"))

Python: 3.12.11
google-generativeai: 0.8.5


c:\Users\rock2\.conda\envs\v312_ga_chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Gemini API Key Securely

In [2]:
load_dotenv()

candidate_vars = ["GOOGLE_GEMINI_KEY", "GEMINI_API_KEY"]
api_key = None
key_source = None

for var in candidate_vars:
    value = os.getenv(var)
    if value and value.strip():
        api_key = value.strip()
        key_source = var
        break

if not api_key:
    raise RuntimeError(
        "Gemini API key not found. Set GOOGLE_GEMINI_KEY or GEMINI_API_KEY in environment or .env."
    )

print(f"Loaded key from: {key_source}")
print(f"Key prefix: {api_key[:6]}... (masked)")

Loaded key from: GOOGLE_GEMINI_KEY
Key prefix: AIzaSy... (masked)


## 3. Initialize Gemini Client

In [7]:
preferred_models = [
    "models/gemini-3.6-flash",
    "models/gemini-3-flash-preview",
    "models/gemini-2.5-flash",
    "models/gemini-2.0-flash",
    "models/gemini-1.5-flash",
]

genai.configure(api_key=api_key)
available = list(genai.list_models())
compatible_names = [
    m.name for m in available
    if "generateContent" in getattr(m, "supported_generation_methods", [])
]

selected = next((name for name in preferred_models if name in compatible_names), None)
if not selected:
    selected = compatible_names[0] if compatible_names else None

if not selected:
    raise RuntimeError("No generateContent-compatible Gemini models were found for this API key.")

test_model_name = selected
model = genai.GenerativeModel(test_model_name)
print(f"Gemini client initialized. Model ready: {test_model_name}")

Gemini client initialized. Model ready: models/gemini-3.6-flash


## 4. Run a Smoke Test Prompt

In [8]:
prompt = "Return exactly: GEMINI_TEST_OK"
raw_response = None
response_text = ""
latency_seconds = None
request_utc = datetime.now(timezone.utc).isoformat()

start = time.perf_counter()
try:
    raw_response = model.generate_content(prompt)
    response_text = (getattr(raw_response, "text", "") or "").strip()
finally:
    latency_seconds = time.perf_counter() - start

print("Raw response type:", type(raw_response).__name__)
print("Response text:", response_text)

Raw response type: GenerateContentResponse
Response text: GEMINI_TEST_OK


## 5. Add Programmatic Verification Checks

In [9]:
checks = {
    "response_object_exists": raw_response is not None,
    "response_text_non_empty": isinstance(response_text, str) and len(response_text) > 0,
    "contains_expected_token": "GEMINI_TEST_OK" in response_text,
}

assert checks["response_object_exists"], "No response object returned from Gemini API"
assert checks["response_text_non_empty"], "Gemini response text is empty"

overall_pass = all(checks.values())
print("CHECK_RESULTS=", json.dumps(checks, indent=2))
print("STATUS=", "PASS" if overall_pass else "WARN")

CHECK_RESULTS= {
  "response_object_exists": true,
  "response_text_non_empty": true,
  "contains_expected_token": true
}
STATUS= PASS


## 6. Handle Common Failure Cases

In [10]:
def run_with_error_handling(model_obj, test_prompt: str):
    try:
        return model_obj.generate_content(test_prompt)
    except Exception as exc:
        msg = str(exc)
        lower = msg.lower()
        if "api key" in lower or "unauthenticated" in lower or "permission" in lower:
            print("FAIL: Invalid or unauthorized API key. Verify GOOGLE_GEMINI_KEY / GEMINI_API_KEY.")
        elif "quota" in lower or "rate" in lower or "429" in lower:
            print("FAIL: Quota or rate limit hit. Check billing/quota and retry later.")
        elif "timeout" in lower or "network" in lower or "connection" in lower:
            print("FAIL: Network issue detected. Check connectivity and retry.")
        else:
            print("FAIL: Unexpected API error:", msg)
        return None

safe_response = run_with_error_handling(model, "Return exactly: GEMINI_TEST_OK")
print("safe_response_exists:", safe_response is not None)

safe_response_exists: True


## 7. Capture and Display Test Metadata

In [11]:
verification_metadata = {
    "request_utc": request_utc,
    "model_name": test_model_name,
    "latency_seconds": round(latency_seconds, 4) if latency_seconds is not None else None,
    "response_length": len(response_text) if isinstance(response_text, str) else 0,
    "status": "PASS" if overall_pass else "WARN",
    "key_source": key_source,
}

print(json.dumps(verification_metadata, indent=2))

{
  "request_utc": "2026-08-21T14:02:35.091715+00:00",
  "model_name": "models/gemini-3.6-flash",
  "latency_seconds": 21.5269,
  "response_length": 14,
  "status": "PASS",
  "key_source": "GOOGLE_GEMINI_KEY"
}
